In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make plots a bit bigger by default
plt.rcParams["figure.figsize"] = (12, 7)


In [ ]:
# -----------------------
# Configure input folders
# -----------------------
# Update these paths to match your machine.
# Windows example: r"data\Channel_Capture_100\iteration_4"
# Linux example:   r"data/Channel_Capture_100/iteration_4"

FOLDER_WITHOUT = r"data\Channel_Capture_100\iteration_4"
FOLDER_WITH     = r"data\Channel_Capture_540\iteration_400"

# Output folder for exported figures
OUT_DIR = Path("figures")
OUT_DIR.mkdir(exist_ok=True, parents=True)

print("Without jammer folder:", FOLDER_WITHOUT)
print("With jammer folder   :", FOLDER_WITH)
print("Output dir           :", OUT_DIR.resolve())


In [ ]:
def _read_csv_two_cols(csv_path: Path) -> np.ndarray:
    """Read a CSV containing (Real, Imag) into a complex vector.
    
    Supports:
      - columns named 'Real' and 'Imaginary' (or 'Imag')
      - or first two numeric columns if headers differ.
    """
    df = pd.read_csv(csv_path)
    cols = [c.lower() for c in df.columns]

    # Try common naming conventions first
    if "real" in cols and ("imaginary" in cols or "imag" in cols):
        real_col = df.columns[cols.index("real")]
        imag_col = df.columns[cols.index("imaginary")] if "imaginary" in cols else df.columns[cols.index("imag")]
        real = df[real_col].to_numpy(dtype=float)
        imag = df[imag_col].to_numpy(dtype=float)
    else:
        # Fallback: use first two columns
        if df.shape[1] < 2:
            raise ValueError(f"CSV must have at least 2 columns (Real, Imag). Got {df.shape[1]} in {csv_path}")
        real = pd.to_numeric(df.iloc[:, 0], errors="coerce").to_numpy(dtype=float)
        imag = pd.to_numeric(df.iloc[:, 1], errors="coerce").to_numpy(dtype=float)

    if np.any(np.isnan(real)) or np.any(np.isnan(imag)):
        raise ValueError(f"Found NaNs after parsing numeric columns in {csv_path}. Check the CSV format.")

    return real + 1j * imag


def load_siso_vector(folder: str | os.PathLike, pick: str = "first") -> tuple[Path, np.ndarray]:
    """Load ONE SISO CSI vector from a folder of CSV files.
    
    Parameters
    ----------
    folder : path to folder containing CSV files
    pick   : 'first' (default) or 'largest' (by number of rows)
    
    Returns
    -------
    (csv_path, h) where h is a complex np.ndarray
    """
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")

    csvs = sorted(folder.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in: {folder}")

    if pick == "largest":
        csv_path = max(csvs, key=lambda p: p.stat().st_size)
    else:
        csv_path = csvs[0]

    h = _read_csv_two_cols(csv_path)
    return csv_path, h


In [ ]:
# -----------------------
# Load SISO vectors
# -----------------------
csv_wo, h_wo = load_siso_vector(FOLDER_WITHOUT, pick="first")
csv_w,  h_w  = load_siso_vector(FOLDER_WITH, pick="first")

print("Loaded WITHOUT jammer:", csv_wo.name, "-> length:", len(h_wo))
print("Loaded WITH jammer   :", csv_w.name,  "-> length:", len(h_w))


In [ ]:
def mag_db(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Magnitude in dB, safe against log(0)."""
    return 20.0 * np.log10(np.maximum(np.abs(x), eps))

def phase_unwrapped(x: np.ndarray) -> np.ndarray:
    """Unwrapped phase in radians."""
    return np.unwrap(np.angle(x))

mag_wo_db = mag_db(h_wo)
mag_w_db  = mag_db(h_w)

ph_wo = phase_unwrapped(h_wo)
ph_w  = phase_unwrapped(h_w)


In [ ]:
# -----------------------
# Plot (SISO) magnitude & phase
# -----------------------
def plot_siso(mag_db_vec, phase_vec, title_prefix: str, out_basename: str | None = None):
    fig, axs = plt.subplots(2, 1, sharex=True, figsize=(12, 7))

    # Magnitude
    axs[0].plot(mag_db_vec, linewidth=2)
    axs[0].set_ylabel("Magnitude (dB)")
    axs[0].grid(True)

    # Phase
    axs[1].plot(phase_vec, linewidth=2)
    axs[1].set_xlabel("Subcarrier Index")
    axs[1].set_ylabel("Phase (rad)")
    axs[1].grid(True)

    fig.suptitle(f"{title_prefix} — SISO CSI", fontsize=16)
    fig.tight_layout()

    if out_basename:
        out_png = OUT_DIR / f"{out_basename}.png"
        out_pdf = OUT_DIR / f"{out_basename}.pdf"
        fig.savefig(out_png, dpi=200, bbox_inches="tight")
        fig.savefig(out_pdf, bbox_inches="tight")
        print("Saved:", out_png, "and", out_pdf)

    plt.show()

plot_siso(mag_wo_db, ph_wo, "Without interference", out_basename="channel_without_jammer_siso")
plot_siso(mag_w_db,  ph_w,  "With interference",    out_basename="channel_with_jammer_siso")


In [ ]:
# -----------------------
# Optional: overlay comparison in one figure
# -----------------------
fig, axs = plt.subplots(2, 1, sharex=True, figsize=(12, 7))

axs[0].plot(mag_wo_db, linewidth=2, label="Without interference")
axs[0].plot(mag_w_db,  linewidth=2, label="With interference")
axs[0].set_ylabel("Magnitude (dB)")
axs[0].grid(True)
axs[0].legend()

axs[1].plot(ph_wo, linewidth=2, label="Without interference")
axs[1].plot(ph_w,  linewidth=2, label="With interference")
axs[1].set_xlabel("Subcarrier Index")
axs[1].set_ylabel("Phase (rad)")
axs[1].grid(True)
axs[1].legend()

fig.suptitle("SISO CSI — Overlay Comparison", fontsize=16)
fig.tight_layout()

out_png = OUT_DIR / "channel_overlay_siso.png"
out_pdf = OUT_DIR / "channel_overlay_siso.pdf"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
fig.savefig(out_pdf, bbox_inches="tight")
print("Saved:", out_png, "and", out_pdf)

plt.show()
